In [0]:
source = spark.table("workspace.default.ieee_train_transaction")

source.select(
    "TransactionID",
    "TransactionAmt",
    "ProductCD"
).orderBy("TransactionID").show(5)

+-------------+--------------+---------+
|TransactionID|TransactionAmt|ProductCD|
+-------------+--------------+---------+
|      2987000|          68.5|        W|
|      2987001|          29.0|        W|
|      2987002|          59.0|        W|
|      2987003|          50.0|        W|
|      2987004|          50.0|        H|
+-------------+--------------+---------+
only showing top 5 rows


In [0]:
INPUT_DIR = "/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing"
dbutils.fs.rm(INPUT_DIR, True)
dbutils.fs.mkdirs(INPUT_DIR)

display(dbutils.fs.ls(INPUT_DIR))

[]

In [0]:
import time
MAX_NEW_BATCHES = 3
BATCH_SIZE = 10
DELAY_SECONDS = 10
last_id = -1
SOURCE_TABLE = "workspace.default.ieee_train_transaction"
source = spark.table(SOURCE_TABLE)
from datetime import datetime
from pyspark.sql import functions as F

replay_start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for batch_index in range(MAX_NEW_BATCHES + 1):

    # 1. Select the next 10 transactions.
    next_batch = (
        source
        .filter(F.col("TransactionID") > last_id)
        .orderBy("TransactionID")
        .limit(BATCH_SIZE)
    )

    # 2. Stop if there are no more records.
    if next_batch.count() == 0:
        print("No more transactions to replay.")
        break

    # 3. Give this batch a synthetic event timestamp.
    # Each batch starts one event-time minute after the previous one.
    event_time = (F.to_timestamp(F.lit(replay_start_time))
    + F.expr(f"INTERVAL {batch_index * 60} SECONDS"))


    ingest_time = datetime.now().replace(microsecond=0)
    next_batch = next_batch.withColumn("event_ts", event_time).withColumn(
        "ts_end",
        F.col("event_ts")
    ).withColumn(
        "ingest_ts",
        F.to_timestamp(F.lit(ingest_time.strftime("%Y-%m-%d %H:%M:%S")))
    )

    # 4. Build a unique directory for this batch.
    batch_name = f"batch_{batch_index:06d}"
    batch_path = f"{INPUT_DIR}/{batch_name}"

    # 5. Refuse to overwrite an existing batch.
    existing_paths = {
        item.path.rstrip("/")
        for item in dbutils.fs.ls(INPUT_DIR)
    }

    if batch_path.rstrip("/") in existing_paths:
        raise RuntimeError(
            f"{batch_path} already exists; refusing to overwrite it."
        )

    # 6. Write the records as Parquet.
    next_batch.write.format("parquet").save(batch_path)
    # next_batch.write.format("parquet").mode("overwrite").save(batch_path)


    # 7. Advance the cursor so the next iteration selects new records.
    last_id = next_batch.agg(
        F.max("TransactionID")
    ).first()[0]

    print(
        f"Wrote {batch_name}: "
        f"{next_batch.count()} rows, "
        f"last TransactionID = {last_id}"
    )

    # 8. Pause before writing the next batch.
    if batch_index < MAX_NEW_BATCHES:
        time.sleep(DELAY_SECONDS)

print("Replay finished.")

Wrote batch_000000: 10 rows, last TransactionID = 2987009
Wrote batch_000001: 10 rows, last TransactionID = 2987019
Wrote batch_000002: 10 rows, last TransactionID = 2987029
Wrote batch_000003: 10 rows, last TransactionID = 2987039
Replay finished.


In [0]:
INPUT_DIR = "/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing"
for item in sorted(dbutils.fs.ls(INPUT_DIR), key=lambda x: x.name):
    if item.name.startswith("batch_"):
        df = spark.read.parquet(item.path)
        print(item.name, "->", df.count(), "rows")

batch_000000/ -> 10 rows
batch_000001/ -> 10 rows
batch_000002/ -> 10 rows
batch_000003/ -> 10 rows


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    IntegerType,
    TimestampType
)

stream_schema = StructType([
    StructField("TransactionID", LongType(), True),
    StructField("TransactionAmt", DoubleType(), True),
    StructField("isFraud", IntegerType(), False),
    StructField("event_ts", TimestampType(), True)
])



In [0]:
INPUT_DIR = (
    "/Volumes/workspace/default/"
    "ieee_fraud_bronze/mp5_landing"
)

transactions_stream = (
    spark.readStream
    .format("parquet")
    .schema(stream_schema)
    .option("recursiveFileLookup", "true")  # Add this line
    .load(INPUT_DIR)
)

transactions_stream.printSchema()
watermarked_stream = transactions_stream.withWatermark(
    "event_ts", "2 minutes"
)

root
 |-- TransactionID: long (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

watermarked_stream = transactions_stream.withWatermark(
    "event_ts",
    "2 minutes"
)
tumbling_60s = (
    watermarked_stream
    .groupBy(
        F.window(F.col("event_ts"), "60 seconds")
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.sum("isFraud").alias("fraud_count")
    )
)
sliding_5m_1m = (
    watermarked_stream
    .groupBy(
        F.window(
            F.col("event_ts"),
            "5 minutes",
            "1 minute"
        )
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.sum("isFraud").alias("fraud_count")
    )
)

In [0]:
BASE_DIR = "/Volumes/workspace/default/ieee_fraud_bronze"
TEST_INPUT_DIR = f"{INPUT_DIR}/batch_000000"
TEST_CHECKPOINT = f"{BASE_DIR}/mp5_checkpoints/source_test"

test_stream = (
    spark.readStream
    .format("parquet")
    .schema(stream_schema)
    .load(TEST_INPUT_DIR)
)

test_query = (
    test_stream.writeStream
    .format("console")
    .option("checkpointLocation", TEST_CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

test_query.awaitTermination()

print("Active:", test_query.isActive)
print("Exception:", test_query.exception())
print("Recent progress:", test_query.recentProgress)

Active: False
Exception: None
Recent progress: [{
    "id": "c26b9b30-2930-4c7f-afa5-6e2facedd467",
    "runId": "c44d23fa-5569-405f-9955-9ec989577813",
    "name": null,
    "timestamp": "2026-10-22T03:12:16.312Z",
    "batchId": 4,
    "batchDuration": 2225,
    "durationMs": {
        "triggerExecution": 2214,
        "queryPlanning": 389,
        "collectSourceMetrics": 3,
        "getBatch": 246,
        "commitOffsets": 103,
        "addBatch": 534,
        "latestOffset": 665,
        "walCommit": 213
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[dbfs:/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing/batch_000000]",
            "startOffset": "{\"logOffset\":3}",
            "endOffset": "{\"logOffset\":4}",
            "latestOffset": null,
            "numInputRows": 10,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 4.4943820224719095,
            "metrics": {}

In [0]:
PARENT_TEST_CHECKPOINT = f"{BASE_DIR}/mp5_checkpoints/parent_test"

parent_stream = (
    spark.readStream
    .format("parquet")
    .schema(stream_schema)
    .option("recursiveFileLookup", "true")
    .load(INPUT_DIR)
)

parent_test_query = (
    parent_stream.writeStream
    .format("console")
    .option("checkpointLocation", PARENT_TEST_CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

parent_test_query.awaitTermination()

progress = parent_test_query.recentProgress
if progress:
    for batch in progress:
        print(
            "Batch ID:", batch.get("batchId"),
            "| Input rows:", batch["sources"][0]["numInputRows"],
            "| Output rows:", batch["sink"]["numOutputRows"]
        )
else:
    print("No progress reports recorded.")

Batch ID: 4 | Input rows: 40 | Output rows: 40


In [0]:
OUTPUT_DIR_FINAL = f"{BASE_DIR}/mp5_outputs_final"
CHECKPOINT_DIR_FINAL = f"{BASE_DIR}/mp5_checkpoints_final"

tumbling_query = (
    tumbling_60s.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{OUTPUT_DIR_FINAL}/tumbling_60s")
    .option("checkpointLocation", f"{CHECKPOINT_DIR_FINAL}/tumbling_60s")
    .trigger(availableNow=True)
    .start()
)

sliding_query = (
    sliding_5m_1m.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{OUTPUT_DIR_FINAL}/sliding_5m_1m")
    .option("checkpointLocation", f"{CHECKPOINT_DIR_FINAL}/sliding_5m_1m")
    .trigger(availableNow=True)
    .start()
)

tumbling_query.awaitTermination()
sliding_query.awaitTermination()

In [0]:
# 1. Read the entire directory of Parquet files into a DataFrame
df_sliding = spark.read.parquet(f"{OUTPUT_DIR_FINAL}/sliding_5m_1m")

# 2. Display the data (choose one option below based on your environment)

# Option A: If you are working in Databricks (best looking interactive table)
display(df_sliding)

# Option B: Standard PySpark console output (prints top 20 rows)
df_sliding.show(truncate=False)


window,transaction_count,total_amount,fraud_count
"List(2026-10-22T01:55:00.000Z, 2026-10-22T02:00:00.000Z)",40,3663.438,0
"List(2026-10-22T03:05:00.000Z, 2026-10-22T03:10:00.000Z)",10,1019.0,0
"List(2024-12-31T23:56:00.000Z, 2025-01-01T00:01:00.000Z)",10,1019.0,0
"List(2026-10-22T01:53:00.000Z, 2026-10-22T01:58:00.000Z)",20,1442.522,0
"List(2024-12-31T23:58:00.000Z, 2025-01-01T00:03:00.000Z)",90,7384.566,0
"List(2026-10-22T01:58:00.000Z, 2026-10-22T02:03:00.000Z)",20,2220.916,0
"List(2026-10-22T01:59:00.000Z, 2026-10-22T02:04:00.000Z)",10,1202.226,0
"List(2026-10-22T01:52:00.000Z, 2026-10-22T01:57:00.000Z)",10,614.7819999999999,0
"List(2026-10-22T01:57:00.000Z, 2026-10-22T02:02:00.000Z)",30,3048.656,0
"List(2025-01-01T00:01:00.000Z, 2025-01-01T00:06:00.000Z)",90,7383.636000000001,0


+------------------------------------------+-----------------+------------------+-----------+
|window                                    |transaction_count|total_amount      |fraud_count|
+------------------------------------------+-----------------+------------------+-----------+
|{2026-10-22 01:55:00, 2026-10-22 02:00:00}|40               |3663.438          |0          |
|{2026-10-22 03:05:00, 2026-10-22 03:10:00}|10               |1019.0            |0          |
|{2024-12-31 23:56:00, 2025-01-01 00:01:00}|10               |1019.0            |0          |
|{2026-10-22 01:53:00, 2026-10-22 01:58:00}|20               |1442.522          |0          |
|{2024-12-31 23:58:00, 2025-01-01 00:03:00}|90               |7384.566          |0          |
|{2026-10-22 01:58:00, 2026-10-22 02:03:00}|20               |2220.916          |0          |
|{2026-10-22 01:59:00, 2026-10-22 02:04:00}|10               |1202.226          |0          |
|{2026-10-22 01:52:00, 2026-10-22 01:57:00}|10              

In [0]:
tumbling_output = spark.read.parquet(
    f"{OUTPUT_DIR_FINAL}/tumbling_60s"
)

sliding_output = spark.read.parquet(
    f"{OUTPUT_DIR_FINAL}/sliding_5m_1m"
)

print("Tumbling window rows:", tumbling_output.count())
print("Sliding window rows:", sliding_output.count())

print("Tumbling results:")
tumbling_output.orderBy("window.start").show(truncate=False)

print("Sliding results:")
sliding_output.orderBy("window.start").show(truncate=False)

Tumbling window rows: 9
Sliding window rows: 17
Tumbling results:
+------------------------------------------+-----------------+------------------+-----------+
|window                                    |transaction_count|total_amount      |fraud_count|
+------------------------------------------+-----------------+------------------+-----------+
|{2025-01-01 00:00:00, 2025-01-01 00:01:00}|10               |1019.0            |0          |
|{2025-01-01 00:01:00, 2025-01-01 00:02:00}|30               |1844.3459999999998|0          |
|{2025-01-01 00:02:00, 2025-01-01 00:03:00}|30               |2483.2200000000003|0          |
|{2025-01-01 00:03:00, 2025-01-01 00:04:00}|30               |3056.0700000000006|0          |
|{2026-10-22 01:56:00, 2026-10-22 01:57:00}|10               |614.7819999999999 |0          |
|{2026-10-22 01:57:00, 2026-10-22 01:58:00}|10               |827.74            |0          |
|{2026-10-22 01:58:00, 2026-10-22 01:59:00}|10               |1018.6900000000002|0      

In [0]:
# 1. Target your final output folder path
target_dir = f"{OUTPUT_DIR_FINAL}/tumbling_60s"

# 2. List all files and loop through them
# (Filtering to ensure we only look at actual data files, ignoring metadata like _SUCCESS)
for file_info in dbutils.fs.ls(target_dir):
    if file_info.name.endswith(".parquet"):
        
        # Manually read just this specific file
        individual_df = spark.read.parquet(file_info.path)
        
        # Print out the file name and its row count
        print(f"File: {file_info.name} -> Contains {individual_df.count()} rows")
        
        # Optional: Uncomment below if you want to inspect a preview of this specific file
        # individual_df.show(5, truncate=False)


File: part-00000-0b2da5b1-8ee7-4211-9b61-d5855e1b3f2a.c000.snappy.parquet -> Contains 0 rows
File: part-00000-564bfcbe-9eae-460f-869b-79c3fd3819d9.c000.snappy.parquet -> Contains 0 rows
File: part-00000-8d929e1e-1e2b-43a2-95a0-00ba2bae037b.c000.snappy.parquet -> Contains 0 rows
File: part-00000-c64bb855-1003-4ecb-a1ee-fd1b651742fc.c000.snappy.parquet -> Contains 0 rows
File: part-00000-f8d4ac6b-d614-466a-857d-4efa30b6bb6e.c000.snappy.parquet -> Contains 0 rows
File: part-00000-fd0b99a9-6227-4f1d-b24c-a56783026dd5.c000.snappy.parquet -> Contains 0 rows
File: part-00053-063d5be7-adba-450b-8277-5f34fe67ac49.c000.snappy.parquet -> Contains 1 rows
File: part-00082-f7987a46-7af1-4ee7-a4eb-18bdb85cd010.c000.snappy.parquet -> Contains 1 rows
File: part-00095-d84ce5b0-08e7-419d-af3a-8606b3705feb.c000.snappy.parquet -> Contains 1 rows
File: part-00102-5c56dab7-8564-4604-8fb6-b9326247d973.c000.snappy.parquet -> Contains 1 rows
File: part-00177-5b0e7b8c-2ec2-4643-8777-ee30a31269ab.c000.snappy.parq

In [0]:
for name, query in [
    ("Tumbling", tumbling_query),
    ("Sliding", sliding_query)
]:
    print(f"\n--- {name} ---")
    for progress in query.recentProgress:
        print("Batch:", progress.get("batchId"))
        print("Input rows:", progress["sources"][0]["numInputRows"])
        print("Event time:", progress.get("eventTime"))
        print("State operators:", progress.get("stateOperators"))


--- Tumbling ---
Batch: 6
Input rows: 40
Event time: {'min': '2026-10-22T03:09:41.000Z', 'avg': '2026-10-22T03:11:11.000Z', 'watermark': '2026-10-22T01:57:44.000Z', 'max': '2026-10-22T03:12:41.000Z'}
State operators: [{
    "operatorName": "stateStoreSave",
    "numRowsTotal": 7,
    "numRowsUpdated": 4,
    "allUpdatesTimeMs": 279,
    "numRowsRemoved": 0,
    "allRemovalsTimeMs": 1524,
    "commitTimeMs": 10340,
    "memoryUsedBytes": 267408719,
    "numRowsDroppedByWatermark": 0,
    "numShufflePartitions": 200,
    "numStateStoreInstances": 200,
    "customMetrics": {
        "rocksdbNumSnapshotsAutoRepaired": 0,
        "rocksdbTotalBytesWrittenByFlush": 0,
        "rocksdbGetLatency": 0,
        "rocksdbNumInternalColFamiliesKeys": 0,
        "SnapshotLastUploaded.partition_157_default": -1,
        "rocksdbCommitCompactLatency": 0,
        "SnapshotLastUploaded.partition_33_default": -1,
        "rocksdbTotalBytesWritten": 356,
        "SnapshotLastUploaded.partition_172_defaul

In [0]:
for name, query in [
    ("Tumbling", tumbling_query),
    ("Sliding", sliding_query)
]:
    print(f"\n{name}")
    for p in query.recentProgress:
        print(
            "batch =", p["batchId"],
            "| watermark =", p.get("eventTime", {}).get("watermark"),
            "| input =", p["sources"][0]["numInputRows"]
        )


Tumbling
batch = 6 | watermark = 2026-10-22T01:57:44.000Z | input = 40
batch = 7 | watermark = 2026-10-22T03:10:41.000Z | input = 0

Sliding
batch = 6 | watermark = 2026-10-22T01:57:44.000Z | input = 40
batch = 7 | watermark = 2026-10-22T03:10:41.000Z | input = 0


In [0]:
for name, query in [
    ("Tumbling", tumbling_query),
    ("Sliding", sliding_query)
]:
    print(f"\n--- {name} ---")

    for p in query.recentProgress:
        print("Batch:", p["batchId"])
        print("Watermark:", p.get("eventTime", {}).get("watermark"))

        for state in p.get("stateOperators", []):
            print("State rows:", state.get("numRowsTotal"))
            print("Updated:", state.get("numRowsUpdated"))
            print("Removed:", state.get("numRowsRemoved"))
            print("Dropped by watermark:", state.get("numRowsDroppedByWatermark"))


--- Tumbling ---
Batch: 6
Watermark: 2026-10-22T01:57:44.000Z
State rows: 7
Updated: 4
Removed: 0
Dropped by watermark: 0
Batch: 7
Watermark: 2026-10-22T03:10:41.000Z
State rows: 3
Updated: 0
Removed: 4
Dropped by watermark: 0

--- Sliding ---
Batch: 6
Watermark: 2026-10-22T01:57:44.000Z
State rows: 15
Updated: 8
Removed: 0
Dropped by watermark: 0
Batch: 7
Watermark: 2026-10-22T03:10:41.000Z
State rows: 7
Updated: 0
Removed: 8
Dropped by watermark: 0


In [0]:
from pyspark.sql import functions as F

LANDING_DIR = "/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing"

# List the batch directories
display(dbutils.fs.ls(LANDING_DIR))

# Read the Parquet replay batches
landing_df = (
    spark.read
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(LANDING_DIR)
)

# Check the schema and sample records
landing_df.printSchema()
display(landing_df.limit(10))

# Check row count and available timestamp fields
print("Total rows:", landing_df.count())
print("Timestamp fields:", [
    c for c in landing_df.columns
    if "ts" in c.lower() or "time" in c.lower()
])

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing/batch_000000/,batch_000000/,0,1792639084835
dbfs:/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing/batch_000001/,batch_000001/,0,1792639084835
dbfs:/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing/batch_000002/,batch_000002/,0,1792639084835
dbfs:/Volumes/workspace/default/ieee_fraud_bronze/mp5_landing/batch_000003/,batch_000003/,0,1792639084835


root
 |-- TransactionID: long (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- TransactionDT: long (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- ProductCD: string (nullable = true)
 |-- card1: string (nullable = true)
 |-- card2: string (nullable = true)
 |-- card3: string (nullable = true)
 |-- card4: string (nullable = true)
 |-- card5: string (nullable = true)
 |-- card6: string (nullable = true)
 |-- addr1: string (nullable = true)
 |-- addr2: string (nullable = true)
 |-- dist1: string (nullable = true)
 |-- dist2: string (nullable = true)
 |-- P_emaildomain: string (nullable = true)
 |-- R_emaildomain: string (nullable = true)
 |-- C1: string (nullable = true)
 |-- C2: string (nullable = true)
 |-- C3: string (nullable = true)
 |-- C4: string (nullable = true)
 |-- C5: string (nullable = true)
 |-- C6: string (nullable = true)
 |-- C7: string (nullable = true)
 |-- C8: string (nullable = true)
 |-- C9: string (nullable = true)
 |-- C10: string

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,event_ts,ts_end,ingest_ts
2987010,0,86549,75.887,C,16496,352.0,117.0,mastercard,134.0,credit,null,null,null,null,gmail.com,gmail.com,1.0,4.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,83.0,0.0,0.0,null,0.0,0.0,0.0,0.0,null,null,null,M0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,0.0,0.0,1.0,2.0,2.0,0.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,0.0,0.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.0,3.0,3.0,4.0,4.0,2.0,1.0,2.0,2.0,4.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,166.21539306640625,166.21539306640625,166.21539306640625,90.3279037475586,31.84129905700684,90.3279037475586,90.3279037475586,90.3279037475586,90.3279037475586,0.0,0.0,0.0,75.88749694824219,75.88749694824219,75.88749694824219,3.0,3.0,3.0,3.0,4.0,4.0,2.0,2.0,2.0,0.0,2.0,4.0,4.0,4.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,166.21539306640625,166.21539306640625,166.21539306640625,90.3279037475586,90.3279037475586,90.3279037475586,0.0,90.3279037475586,90.3279037475586,90.3279037475586,0.0,0.0,0.0,75.88749694824219,75.88749694824219,75.88749694824219,3.0,3.0,3.0,4.0,4.0,2.0,2.0,1.0,2.0,2.0,2.0,4.0,4.0,4.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,166.21539306640625,166.21539306640625,166.21539306640625,90.3279037475586,90.3279037475586,31.84129905700684,90.3279037475586,90.3279037475586,90.3279037475586,90.3279037475586,0.0,0.0,0.0,75.88749694824219,75.88749694824219,75.88749694824219,null,null,n

Total rows: 40
Timestamp fields: ['event_ts', 'ts_end', 'ingest_ts']


In [0]:
from pyspark.sql import functions as F
import math

calibration_df = (
    spark.read
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(INPUT_DIR)
)

required = {"event_ts", "ts_end", "ingest_ts"}
missing = required - set(calibration_df.columns)

if missing:
    raise ValueError(f"Missing required timestamp columns: {missing}")

print("Rows:", calibration_df.count())
calibration_df.select(
    "TransactionID", "event_ts", "ts_end", "ingest_ts"
).orderBy("TransactionID").show(10, truncate=False)

Rows: 40
+-------------+-------------------+-------------------+-------------------+
|TransactionID|event_ts           |ts_end             |ingest_ts          |
+-------------+-------------------+-------------------+-------------------+
|2987000      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987001      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987002      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987003      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987004      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987005      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987006      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987007      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987008      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
|2987009      |2026-10-22 03:09:41|2026-10-22 03:09:41|2026-10-22 03:09:42|
+--

In [0]:
lateness_df = calibration_df.withColumn(
    "lateness_seconds",
    F.col("ingest_ts").cast("long")
    - F.col("ts_end").cast("long")
)

percentiles = lateness_df.agg(
    F.count("*").alias("record_count"),
    F.expr(
        "percentile_approx(lateness_seconds, 0.50, 10000)"
    ).alias("p50_seconds"),
    F.expr(
        "percentile_approx(lateness_seconds, 0.95, 10000)"
    ).alias("p95_seconds"),
    F.expr(
        "percentile_approx(lateness_seconds, 0.99, 10000)"
    ).alias("p99_seconds"),
    F.sum(
        F.when(F.col("lateness_seconds") < 0, 1).otherwise(0)
    ).alias("negative_lateness_count")
)

percentiles.show(truncate=False)

+------------+-----------+-----------+-----------+-----------------------+
|record_count|p50_seconds|p95_seconds|p99_seconds|negative_lateness_count|
+------------+-----------+-----------+-----------+-----------------------+
|40          |-83        |1          |1          |30                     |
+------------+-----------+-----------+-----------+-----------------------+



In [0]:
stats = percentiles.first()
p99 = stats["p99_seconds"]

watermark_seconds = max(60, math.ceil(p99))
watermark_delay = f"{watermark_seconds} seconds"

print("p99 lateness:", p99, "seconds")
print("Selected watermark delay:", watermark_delay)

p99 lateness: 1 seconds
Selected watermark delay: 60 seconds


In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType,
    IntegerType, TimestampType
)

FAULT_BASE = f"{BASE_DIR}/mp5_fault_test"
FAULT_OUTPUT = f"{FAULT_BASE}/outputs"
FAULT_CHECKPOINT = f"{FAULT_BASE}/checkpoints"

FAULT_INPUT = INPUT_DIR  # Your existing calibration landing data

fault_schema = StructType([
    StructField("TransactionID", LongType(), True),
    StructField("TransactionAmt", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("event_ts", TimestampType(), True)
])

fault_stream = (
    spark.readStream
    .format("parquet")
    .schema(fault_schema)
    .option("recursiveFileLookup", "true")
    .option("maxFilesPerTrigger", 1)
    .load(FAULT_INPUT)
)

print("Fault-test input configured.")

Fault-test input configured.


In [0]:
fault_watermarked = fault_stream.withWatermark(
    "event_ts", "60 seconds"
)

fault_tumbling = (
    fault_watermarked
    .groupBy(F.window("event_ts", "60 seconds"))
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.sum("isFraud").alias("fraud_count")
    )
)

fault_query = (
    fault_tumbling.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{FAULT_OUTPUT}/tumbling_60s")
    .option(
        "checkpointLocation",
        f"{FAULT_CHECKPOINT}/tumbling_60s"
    )
    .trigger(availableNow=True)
    .start()
)

print("Query started:", fault_query.id)
print("Status:", fault_query.status)

Query started: d59ab84d-d592-477e-9c6a-07ebc842554c
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
import time

# Start the streaming query
fault_query = (
    fault_tumbling.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{FAULT_OUTPUT}/tumbling_60s")
    .option("checkpointLocation", f"{FAULT_CHECKPOINT}/tumbling_60s")
    .trigger(availableNow=True)
    .start()
)

print("Query started. Waiting 18 seconds...")
time.sleep(18)

print("Active before interruption:", fault_query.isActive)
print("Recent progress:", fault_query.recentProgress)

if fault_query.isActive:
    fault_query.stop()
    print("Stop requested.")
else:
    print("Query already finished; no interruption occurred.")

print("Active after interruption:", fault_query.isActive)
print("Exception:", fault_query.exception())

Query started. Waiting 18 seconds...
Active before interruption: True
Recent progress: [{
    "id": "18531a79-726d-4544-b608-78d819782807",
    "runId": "f3bc2e1a-0919-4f0c-a4d3-d612220ee9e9",
    "name": null,
    "timestamp": "2026-10-22T03:44:10.350Z",
    "batchId": 0,
    "batchDuration": 11302,
    "durationMs": {
        "triggerExecution": 11302,
        "queryPlanning": 700,
        "collectSourceMetrics": 0,
        "getBatch": 163,
        "commitOffsets": 115,
        "addBatch": 10152,
        "latestOffset": 84,
        "walCommit": 163
    },
    "eventTime": {
        "min": "2026-10-22T03:09:41.000Z",
        "avg": "2026-10-22T03:09:41.000Z",
        "watermark": "1970-01-01T00:00:00.000Z",
        "max": "2026-10-22T03:09:41.000Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 1,
            "numRowsUpdated": 1,
            "allUpdatesTimeMs": 61,
            "numRowsRemoved": 0,
            "allRemo

In [0]:
fault_query = (
    fault_tumbling.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{FAULT_OUTPUT}/tumbling_60s")
    .option("checkpointLocation", f"{FAULT_CHECKPOINT}/tumbling_60s")
    .trigger(availableNow=True)
    .start()
)

fault_query.awaitTermination()

print("Restart completed.")
print("Exception:", fault_query.exception())
print("Recent progress:", fault_query.recentProgress)

Restart completed.
Exception: None
Recent progress: [{
    "id": "18531a79-726d-4544-b608-78d819782807",
    "runId": "c6374d48-b88f-4caf-bdb4-6ae7d08dde12",
    "name": null,
    "timestamp": "2026-10-22T03:47:35.632Z",
    "batchId": 2,
    "batchDuration": 16141,
    "durationMs": {
        "triggerExecution": 16140,
        "queryPlanning": 929,
        "collectSourceMetrics": 0,
        "getBatch": 127,
        "commitOffsets": 73,
        "addBatch": 14946
    },
    "eventTime": {
        "min": "2026-10-22T03:11:41.000Z",
        "avg": "2026-10-22T03:11:41.000Z",
        "watermark": "2026-10-22T03:09:41.000Z",
        "max": "2026-10-22T03:11:41.000Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 3,
            "numRowsUpdated": 1,
            "allUpdatesTimeMs": 107,
            "numRowsRemoved": 0,
            "allRemovalsTimeMs": 1674,
            "commitTimeMs": 9121,
            "memoryUsedBytes": 26147

In [0]:
output_path = f"{FAULT_OUTPUT}/tumbling_60s"

out = spark.read.parquet(output_path)

print("Output rows:", out.count())
print("Distinct window keys:",
      out.select("window.start", "window.end").distinct().count())

print("Duplicate window keys:")
out.groupBy("window.start", "window.end") \
   .count() \
   .filter("count > 1") \
   .show(truncate=False)

print("Output window aggregates:")
out.orderBy("window.start").show(truncate=False)

Output rows: 2
Distinct window keys: 2
Duplicate window keys:
+-----+---+-----+
|start|end|count|
+-----+---+-----+
+-----+---+-----+

Output window aggregates:
+------------------------------------------+-----------------+-----------------+-----------+
|window                                    |transaction_count|total_amount     |fraud_count|
+------------------------------------------+-----------------+-----------------+-----------+
|{2026-10-22 03:09:00, 2026-10-22 03:10:00}|10               |1019.0           |0          |
|{2026-10-22 03:10:00, 2026-10-22 03:11:00}|10               |614.7819999999999|0          |
+------------------------------------------+-----------------+-----------------+-----------+



In [0]:
# Read the same input used by the streaming query
batch_input = (
    spark.read
    .schema(fault_schema)
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(FAULT_INPUT)
)

print("Input transaction rows:", batch_input.count())

expected = (
    batch_input
    .groupBy(F.window("event_ts", "60 seconds"))
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.sum("isFraud").alias("fraud_count")
    )
)

print("Expected window aggregates:")
expected.orderBy("window.start").show(truncate=False)

print("Streaming output aggregates:")
out.orderBy("window.start").show(truncate=False)


Input transaction rows: 40
Expected window aggregates:
+------------------------------------------+-----------------+------------------+-----------+
|window                                    |transaction_count|total_amount      |fraud_count|
+------------------------------------------+-----------------+------------------+-----------+
|{2026-10-22 03:09:00, 2026-10-22 03:10:00}|10               |1019.0            |0          |
|{2026-10-22 03:10:00, 2026-10-22 03:11:00}|10               |614.7819999999999 |0          |
|{2026-10-22 03:11:00, 2026-10-22 03:12:00}|10               |827.74            |0          |
|{2026-10-22 03:12:00, 2026-10-22 03:13:00}|10               |1018.6900000000002|0          |
+------------------------------------------+-----------------+------------------+-----------+

Streaming output aggregates:
+------------------------------------------+-----------------+-----------------+-----------+
|window                                    |transaction_count|total_am

In [0]:
fault_query = (
    fault_tumbling.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", f"{FAULT_OUTPUT}/tumbling_60s")
    .option("checkpointLocation", f"{FAULT_CHECKPOINT}/tumbling_60s")
    .trigger(availableNow=True)
    .start()
)

fault_query.awaitTermination()

print("Exception:", fault_query.exception())

Exception: None


In [0]:
out = spark.read.parquet(f"{FAULT_OUTPUT}/tumbling_60s")

print("Streaming output rows:", out.count())
print(
    "Distinct window keys:",
    out.select("window.start", "window.end").distinct().count()
)

out.orderBy("window.start").show(truncate=False)

Streaming output rows: 2
Distinct window keys: 2
+------------------------------------------+-----------------+-----------------+-----------+
|window                                    |transaction_count|total_amount     |fraud_count|
+------------------------------------------+-----------------+-----------------+-----------+
|{2026-10-22 03:09:00, 2026-10-22 03:10:00}|10               |1019.0           |0          |
|{2026-10-22 03:10:00, 2026-10-22 03:11:00}|10               |614.7819999999999|0          |
+------------------------------------------+-----------------+-----------------+-----------+

